# Translator Test

Loads the checkpoint produced by `python -m translator.train` (a pure-NumPy Transformer, no PyTorch, trained with hand-written backprop -- see `translator/layers.py`, `translator/attention.py`, `translator/model.py`) and runs it on many example English sentences to see what it actually learned.

**Honest expectations:** the model in `translator/train.py` is deliberately tiny (`d_model=64`, 1 layer, ~300 short sentence pairs, character-level tokenization) so that training with pure-NumPy backprop finishes in a few minutes on a CPU. It is not going to produce fluent translation -- the goal is to see the architecture actually *learn* (loss going down, training sentences becoming recognizable), which the forward-only code in `src/` structurally cannot do (see `docs/known-issues.md` #4). Two groups of examples below:

1. **Seen sentences** -- pulled from the training set itself, compared against their real target. These should look noticeably better than random.
2. **Unseen sentences** -- typed by hand, never seen during training. These are the real test of generalization, and with this little data they will likely be rough or nonsensical -- that's expected, not a bug.

In [1]:
import os, sys
sys.path.append(os.path.abspath(".."))

from translator.checkpoint import load_checkpoint
from translator.data import clean_sentence, load_dataset
from translator.translate import greedy_decode

CHECKPOINT_PATH = os.path.join("checkpoint.pkl")
model, hyperparams, english_vocab, vietnamese_vocab = load_checkpoint(CHECKPOINT_PATH)
english_vocabulary, index_to_english, english_char_to_index = english_vocab
vietnamese_vocabulary, index_to_vietnamese, vietnamese_char_to_index = vietnamese_vocab
max_sequence_length = hyperparams["max_sequence_length"]

print("Hyperparameters:", hyperparams)
print("English vocab size:", len(english_vocabulary))
print("Vietnamese vocab size:", len(vietnamese_vocabulary))
print("Total parameters:", sum(p.data.size for p in model.parameters()))

Hyperparameters: {'d_model': 64, 'ffn_hidden': 128, 'num_heads': 4, 'num_layers': 1, 'drop_prob': 0.1, 'max_sequence_length': 32}
English vocab size: 36
Vietnamese vocab size: 96
Total parameters: 98400


In [2]:
def safe_translate(raw_sentence):
    """Cleans/lowercases input the same way training data was cleaned, drops
    any character the model's vocabulary has never seen (it would otherwise
    KeyError on tokenization), and runs greedy decoding."""
    cleaned = clean_sentence(raw_sentence.lower())
    filtered = "".join(ch for ch in cleaned if ch in english_char_to_index)
    dropped = sorted(set(cleaned) - set(filtered))
    if len(filtered) > max_sequence_length - 2:
        filtered = filtered[: max_sequence_length - 2]
    prediction = greedy_decode(model, filtered, max_sequence_length, index_to_vietnamese)
    return prediction, filtered, dropped


def show_translation(raw_sentence, target=None):
    prediction, filtered, dropped = safe_translate(raw_sentence)
    print(f"EN:         {raw_sentence}")
    if filtered != clean_sentence(raw_sentence.lower()):
        print(f"  (used):   {filtered}")
    if dropped:
        print(f"  (dropped out-of-vocabulary characters: {dropped})")
    print(f"VI (pred):  {prediction}")
    if target is not None:
        print(f"VI (true):  {target}")
    print("-" * 60)

## 1. Seen sentences (from the training set)

Re-loads the same slice of `data/train.en`/`data/train.vi` that `translator/train.py` trained on (same `num_sentences`/`max_sentence_length`, so this is exactly the training set), and translates the first several of them.

In [3]:
english_sentences, vietnamese_sentences, _, _, _ = load_dataset(
    num_sentences=300, max_sentence_length=max_sequence_length - 2
)

N_SEEN_EXAMPLES = 15
for en, vi in zip(english_sentences[:N_SEEN_EXAMPLES], vietnamese_sentences[:N_SEEN_EXAMPLES]):
    show_translation(en, target=vi)

EN:         thank you very much
VI (pred):  cảm ơn các cảm cảm các
VI (true):  cám ơn rất nhiều
------------------------------------------------------------
EN:         it was terribly dangerous
VI (pred):  đó là thì thiều thi thiều
VI (true):  điều này thực sự nguy hiểm
------------------------------------------------------------
EN:         where will we take it
VI (pred):  đó là thì thiề thì thiều
VI (true):  chúng ta sẽ đưa nó đến đâu
------------------------------------------------------------
EN:         the outcome immediate
VI (pred):  đó là thì thiều thi nhiều
VI (true):  có kết quả ngay lập tức
------------------------------------------------------------
EN:         who was right who wrong
VI (pred):  đó là thì thiều thì thiều
VI (true):  ai đã đúng ai sai
------------------------------------------------------------
EN:         was the tale told well
VI (pred):  đó là thì thiều thì thiều
VI (true):  câu chuyện kể có hay không
--------------------------------------------------

EN:         thank you
VI (pred):  cảm ơn ơn
VI (true):  cảm ơn các bạn
------------------------------------------------------------
EN:         but what about that guy
VI (pred):  có là thì tôi nó thi nhiệt
VI (true):  nhưng còn người này thì sao
------------------------------------------------------------
EN:         i thought so
VI (pred):  đó là đã tôi đã đây
VI (true):  tôi nghĩ là tốt
------------------------------------------------------------
EN:         the press started calling us
VI (pred):  đó là thì thiều thi thiều
VI (true):  báo giới bắt đầu gọi chúng tôi
------------------------------------------------------------
EN:         looked like a ladybug right
VI (pred):  có là thọ thi nhể thiều
VI (true):  trông như một con bọ hung nhỉ
------------------------------------------------------------
EN:         you can do a lot of things
VI (pred):  đó nhúng thôi nó thi nhiệt
VI (true):  bạn có thể làm rất nhiều thứ
------------------------------------------------------------
EN: 

EN:         thank you very much
VI (pred):  cảm ơn các cảm cảm các
VI (true):  cám ơn rất nhiều
------------------------------------------------------------


### Character-level accuracy on the training set

A quick quantitative signal alongside the qualitative examples above: for each sentence, the fraction of predicted characters (up to the length of whichever of prediction/target is shorter) that match the true translation, averaged across all training sentences.

In [4]:
def char_accuracy(prediction, target):
    if not target:
        return 0.0
    n = min(len(prediction), len(target))
    matches = sum(1 for i in range(n) if prediction[i] == target[i])
    return matches / len(target)


accuracies = []
for en, vi in zip(english_sentences, vietnamese_sentences):
    prediction, _, _ = safe_translate(en)
    accuracies.append(char_accuracy(prediction, vi))

print(f"Mean character accuracy over {len(accuracies)} training sentences: {sum(accuracies)/len(accuracies):.1%}")
print(f"Best: {max(accuracies):.1%}   Worst: {min(accuracies):.1%}")

Mean character accuracy over 300 training sentences: 14.8%
Best: 100.0%   Worst: 0.0%


## 2. Unseen sentences (hand-written, not in the training data)

This is the real generalization test. With ~300 training sentences and a tiny model, don't expect fluent Vietnamese here -- the point is to see *something* related come out (fragments, a plausible-looking character or two) rather than pure noise, and to compare against how much cleaner the seen-sentence outputs above are.

In [5]:
unseen_sentences = [
    "hello",
    "thank you",
    "how are you",
    "i love you",
    "what is your name",
    "good morning",
    "see you tomorrow",
    "this is a test",
    "the weather is nice today",
    "can you help me",
]

for sentence in unseen_sentences:
    show_translation(sentence)

EN:         hello
VI (pred):  đó húng
------------------------------------------------------------
EN:         thank you
VI (pred):  cảm ơn ơn
------------------------------------------------------------
EN:         how are you
VI (pred):  cảm cảm các bạn
------------------------------------------------------------
EN:         i love you
VI (pred):  cảm ơn các bạn
------------------------------------------------------------
EN:         what is your name
VI (pred):  cảm là tôi bạn các nhiều
------------------------------------------------------------
EN:         good morning
VI (pred):  đó là đã tôi đó đây
------------------------------------------------------------
EN:         see you tomorrow
VI (pred):  đó là đã tôi đã đây
------------------------------------------------------------
EN:         this is a test
VI (pred):  đó là đó thôi đó đây
------------------------------------------------------------
EN:         the weather is nice today
VI (pred):  đó là thì thiều thi thiều
-------

EN:         can you help me
VI (pred):  đó là đã tôi đó đây
------------------------------------------------------------


## 3. Try your own sentence

In [6]:
show_translation("where is the nearest train station")

EN:         where is the nearest train station
  (used):   where is the nearest train sta
VI (pred):  đó là thì thi thì thiều
------------------------------------------------------------
